# TinyLlama Classical RAG: FAISS on SQuAD

This notebook implements a complete RAG pipeline for TinyLlama. 
1. **Setup**: Install dependencies.
2. **Data**: Load SQuAD contexts.
3. **Index**: Embed contexts and build FAISS index.
4. **RAG**: Retrieve contexts and generate grounded answers with Similarity Thresholding.

In [1]:
!pip install -q transformers sentence-transformers faiss-cpu datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from datasets import load_dataset

# 1. Load Models
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    device_map="auto"
)
gen_pipeline = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer,
)

print("Loading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
# 2. Prepare Data & Index
dataset = load_dataset("squad", split="train")
contexts = list(set(dataset["context"][:100]))[:20]

print(f"Embedding {len(contexts)} contexts...")
embeddings = embed_model.encode(contexts)
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
index.add(np.array(embeddings).astype("float32"))
print("Indexing complete.")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Embedding 20 contexts...
Indexing complete.


In [6]:
# 3. RAG Functions with Similarity Thresholding
def retrieve(query, top_k=3):
    query_vec = embed_model.encode([query])
    faiss.normalize_L2(query_vec)
    distances, indices = index.search(np.array(query_vec).astype("float32"), top_k)
    
    # Threshold: if similarity is < 0.4, the context is likely irrelevant
    best_score = distances[0][0]
    if best_score < 0.2:
        return [], best_score
        
    return [contexts[i] for i in indices[0]], best_score

def generate_answer(query, context, score):
    if not context:
        return f"NOT FOUND (Similarity score {score:.2f} too low)"
        
    prompt = f"""Context:
{context}

Question: {query}

Instruction: Extract the answer strictly from the Context above. If the Context does not provide the answer, say "NOT FOUND".

Answer:"""
    
    outputs = gen_pipeline(
        prompt, 
        max_new_tokens=50, 
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = outputs[0]["generated_text"].split("Answer:")[-1].strip()
    return answer

In [7]:
# 4. Final Stress Test
stress_queries = [
    "What is the capacity of the stadium?",
    "What is the population of Notre Dame as per the context?",
    "Who is the current Prime Minister of India?"
]

for q in stress_queries:
    print(f"\nQuery: {q}")
    retrieved_docs, score = retrieve(q)
    context_str = "\n".join(retrieved_docs)
    answer = generate_answer(q, context_str, score)
    print(f"RAG Answer: {answer}")

Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: What is the capacity of the stadium?


Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG Answer: The stadium has a capacity of 80,000.

Query: What is the population of Notre Dame as per the context?
RAG Answer: The population of Notre Dame as per the context is 12,179 students, with 8,448 undergraduates, 2,138 graduate and professional, and 1,593

Query: Who is the current Prime Minister of India?
RAG Answer: NOT FOUND (Similarity score 0.07 too low)
